In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Interactive sound selection + labeling tool
-------------------------------------------
Lets the user:
- Audition sounds
- Select 2s segments
- Assign coarse and fine labels
- Automatically normalize, ramp, and save them

Ensures:
- 160 total processed sounds
- 80 Industrial / 80 Nature
"""

import os, json, random
import pandas as pd
import numpy as np
import librosa, soundfile as sf
import IPython.display as ipd
from IPython.display import clear_output

# ========== AUDIO PROCESSING HELPERS ==========
def process_clip(in_path, out_path, start, end, ramp_dur=0.5):
    """Trim, RMS-normalize, apply ramp, and save a 2s segment."""
    y, sr = librosa.load(in_path, sr=None)
    start_samp, end_samp = int(start * sr), int(end * sr)
    clip = y[start_samp:end_samp]
    if len(clip) < sr * 2:
        print("⚠️ Clip shorter than 2s — skipping.")
        return False

    # RMS normalize
    rms = np.sqrt(np.mean(clip**2))
    if rms > 0:
        clip = clip / rms

    # Apply 0.5s ramp
    ramp_samples = int(ramp_dur * sr)
    ramp = np.linspace(0, 1, ramp_samples)
    clip[:ramp_samples] *= ramp
    clip[-ramp_samples:] *= ramp[::-1]

    # Save
    sf.write(out_path, clip, sr)
    return True

In [ ]:
# ========== CONFIGURATION ==========
SOURCE_PATH = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/NaturalSounds"
TO_SAVE_PATH = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/NaturalSounds2025"
BASE_STIM_PATH = "/om/data/public/audioset/wavs/eval_segments_downloads/"
os.makedirs(TO_SAVE_PATH, exist_ok=True)

TARGET_TOTAL = 160
TARGET_PER_CATEGORY = TARGET_TOTAL // 2

# Metadata file should point to full paths or relative paths under SOURCE_PATH
meta = pd.read_csv("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/NaturalSounds/metadata.csv")

# If metadata only lists filenames, prepend SOURCE_PATH
#if not meta["filepath"].iloc[0].startswith("/"):
meta["filepath"] = BASE_STIM_PATH + meta["filepath"].apply(lambda x: x)

all_files = meta["filepath"].tolist()
random.shuffle(all_files)
print(all_files[0])

In [ ]:
# ========== LOAD PROGRESS ==========
json_path = os.path.join(TO_SAVE_PATH, "filenames.json")
try:
    with open(json_path, "r") as f:
        processed = json.load(f)
        clip_num = len(processed)  # use for naming saved clips
        print(f"Resuming from index {clip_num}")
except FileNotFoundError:
    print("Starting fresh")
    processed, clip_num = [], 0

idx = 0  # index through all files

# Helper counters
n_industrial = sum(1 for x in processed if x.get("coarse_label") == "industrial")
n_nature = sum(1 for x in processed if x.get("coarse_label") == "nature")

# ========== MAIN LOOP ==========
while len(processed) < TARGET_TOTAL and idx < len(all_files):
    clear_output(wait=True)
    file_path = all_files[idx]

    print(f"\nSelected {len(processed)}/{TARGET_TOTAL} clips "
          f"({n_industrial} industrial, {n_nature} nature)")
    ipd.display(ipd.Audio(file_path))
    print(file_path)

    # Ask if user wants to keep this sound
    keep = input("Keep this sound? (y/n): ").strip().lower()
    if keep not in ["y", "yes"]:
        print("⏭️ Skipping this sound.")
        idx += 1
        continue

    # Ask for coarse label
    coarse_label = input("Coarse label (Industrial / Nature): ").strip().lower()
    if coarse_label not in ["industrial", "nature"]:
        print("Invalid label — try again.")
        continue

    # Enforce category balance
    if coarse_label == "industrial" and n_industrial >= TARGET_PER_CATEGORY:
        print("Already have enough Industrial sounds.")
        idx += 1
        continue
    if coarse_label == "nature" and n_nature >= TARGET_PER_CATEGORY:
        print("Already have enough Nature sounds.")
        idx += 1
        continue

    fine_label = input("Fine label (free text): ").strip()
    onset = float(input("Enter onset time (in seconds): "))

    # Process clip
    out_fname = f"mem_stim_num_{clip_num}.wav"   # use clip_num, not idx
    out_path = os.path.join(TO_SAVE_PATH, out_fname)
    success = process_clip(file_path, out_path, onset, onset + 2)

    if success:
        entry = {
            "orig_path": file_path,
            "stim_path": out_path,
            "filename": out_fname,
            "onset": onset,
            "coarse_label": coarse_label,
            "fine_label": fine_label,
        }
        processed.append(entry)

        # Save progress
        with open(json_path, "w") as f:
            json.dump(processed, f, indent=2)

        # Update counters and clip number
        if coarse_label == "industrial":
            n_industrial += 1
        else:
            n_nature += 1
        clip_num += 1  # increment only when something is saved
        print(f"✅ Saved clip {out_fname}")
    else:
        print("⚠️ Skipped due to invalid segment or short duration.")

    idx += 1  # always move to next source file

    # Stop when quotas reached
    if n_industrial >= TARGET_PER_CATEGORY and n_nature >= TARGET_PER_CATEGORY:
        print("🎯 Completed 160 clips (80 Industrial, 80 Nature).")
        break

In [ ]:
# ========== HANDLE REPLACEMENTS ==========
import datetime
import re

print("\n🔎 Checking for replacement clips...")

pattern = re.compile(r"replace(\d+)_([a-z]+)_(.+)\.wav", re.IGNORECASE)
replace_files = [
    f for f in os.listdir(TO_SAVE_PATH)
    if f.lower().startswith("replace") and f.lower().endswith(".wav")
]

if not replace_files:
    print("No replacement files found.")
else:
    # --- Backup current JSON ---
    timestamp = datetime.datetime.now().strftime("%Y-%m-%dT%H-%M-%S")
    backup_path = os.path.join(
        TO_SAVE_PATH, f"filenames_backup_{timestamp}.json"
    )
    with open(backup_path, "w") as f:
        json.dump(processed, f, indent=2)
    print(f"🧷 Backup saved to {backup_path}")

    # --- Process replacements ---
    print(f"Found {len(replace_files)} replacement file(s):")
    for fname in replace_files:
        print(" •", fname)

    for fname in replace_files:
        match = pattern.match(fname)
        if not match:
            print(f"⚠️ Skipping {fname} (invalid filename pattern)")
            continue

        idx_str, coarse_label, fine_label = match.groups()
        idx = int(idx_str)

        if idx >= len(processed):
            print(f"❌ Index {idx} out of range (len={len(processed)}) — skipping.")
            continue

        entry = processed[idx]
        replace_path = os.path.join(TO_SAVE_PATH, fname)

        # Standardized target path
        new_fname = f"mem_stim_num_{idx}.wav"
        new_path = os.path.join(TO_SAVE_PATH, new_fname)

        print(f"\n🔁 Replacing entry {idx}:")
        print(f"   Old file: {os.path.basename(entry['stim_path'])}")
        print(f"   New file: {fname} → {new_fname}")

        # Option to specify onset for processing
        try:
            onset = float(input(f"Enter onset time for {fname} (seconds): ").strip())
        except ValueError:
            print("⚠️ Invalid onset. Defaulting to 0.0")
            onset = 0.0

        # Process new clip into the target name
        success = process_clip(replace_path, new_path, onset, onset + 2)
        if not success:
            print(f"⚠️ Skipped replacement {fname} — processing failed.")
            continue

        # Remove original replace* file after processing
        os.remove(replace_path)

        # Update metadata
        entry.update({
            "stim_path": new_path,
            "filename": new_fname,
            "coarse_label": coarse_label.lower(),
            "fine_label": fine_label.replace("_", " "),
            "onset": onset,
            "replaced": True,
            "replacement_date": timestamp,
        })

        print(f"✅ Successfully replaced mem_stim_num_{idx}.wav")

    # --- Save updated JSON ---
    with open(json_path, "w") as f:
        json.dump(processed, f, indent=2)

    print(f"\n✅ Updated {json_path} and applied all replacements.")
    print("🧩 All replacements processed and normalized successfully.")

In [ ]:
# ========== POST-HOC ORIG_PATH FIX ==========
import datetime, json, os

# Backup before editing
timestamp = datetime.datetime.now().strftime("%Y-%m-%dT%H-%M-%S")
backup_path = os.path.join(TO_SAVE_PATH, f"filenames_backup_fix_{timestamp}.json")
with open(backup_path, "w") as f:
    json.dump(processed, f, indent=2)
print(f"🧷 Backup saved to {backup_path}")

# Fix all replaced entries
n_fixed = 0
for entry in processed:
    if entry.get("replaced") and os.path.exists(entry["stim_path"]):
        # If orig_path still points to old /om/ data, redirect it to the processed file
        entry["orig_path"] = entry["stim_path"]
        n_fixed += 1

# Save updates
with open(json_path, "w") as f:
    json.dump(processed, f, indent=2)

print(f"✅ Fixed orig_path for {n_fixed} replaced entries in {json_path}")

In [ ]:
# ========== FIX BAD ORIG_PATH PREFIXES ==========
import json, os, datetime

# Backup first (always)
timestamp = datetime.datetime.now().strftime("%Y-%m-%dT%H-%M-%S")
backup_path = os.path.join(TO_SAVE_PATH, f"filenames_backup_prefixfix_{timestamp}.json")
with open(backup_path, "w") as f:
    json.dump(processed, f, indent=2)
print(f"🧷 Backup saved to {backup_path}")

# Prefix to remove
BAD_PREFIX = "/Users/bjm/Documents/other/eval_segments_downloads_0/"

# Iterate and fix
n_fixed = 0
for entry in processed:
    path = entry.get("orig_path", "")
    if path.startswith(BAD_PREFIX):
        new_path = path.replace(BAD_PREFIX, "/", 1)  # remove prefix once
        entry["orig_path"] = new_path
        n_fixed += 1

# Save changes
with open(json_path, "w") as f:
    json.dump(processed, f, indent=2)

print(f"✅ Fixed {n_fixed} malformed orig_path entries in {json_path}")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Split NaturalSounds2025 dataset into two balanced halves (p1 and p2)
Each half will contain equal numbers of 'industrial' and 'nature' sounds.
"""

import os
import json
import random
import shutil
from pathlib import Path

# ====================== CONFIG ======================
meta_path = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/NaturalSounds2025/filenames.json"
src_dir = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/NaturalSounds2025/"
dst_p1 = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/NaturalSounds2025_p1/"
dst_p2 = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/NaturalSounds2025_p2/"
seed = 42  # for reproducibility

# ====================================================
random.seed(seed)

# --- Load metadata ---
with open(meta_path, "r") as f:
    meta = json.load(f)

# --- Group by coarse_label ---
industrial = [x for x in meta if x["coarse_label"] == "industrial"]
nature = [x for x in meta if x["coarse_label"] == "nature"]

print(f"Loaded {len(meta)} total items: {len(industrial)} industrial, {len(nature)} nature.")

# --- Shuffle within each category ---
random.shuffle(industrial)
random.shuffle(nature)

# --- Split each half ---
n_ind_half = len(industrial) // 2
n_nat_half = len(nature) // 2

industrial_p1, industrial_p2 = industrial[:n_ind_half], industrial[n_ind_half:]
nature_p1, nature_p2 = nature[:n_nat_half], nature[n_nat_half:]

meta_p1 = industrial_p1 + nature_p1
meta_p2 = industrial_p2 + nature_p2

# --- Shuffle within each partition (optional aesthetic) ---
random.shuffle(meta_p1)
random.shuffle(meta_p2)

# --- Create output dirs ---
os.makedirs(dst_p1, exist_ok=True)
os.makedirs(dst_p2, exist_ok=True)

# --- Copy files and update stim paths ---
def copy_and_update(meta_subset, dst_dir):
    updated = []
    for i, entry in enumerate(meta_subset):
        src = entry["stim_path"]
        new_name = f"mem_stim_{i}.wav"  # sequential renumbering
        dst = os.path.join(dst_dir, new_name)
        shutil.copy2(src, dst)

        new_entry = entry.copy()
        new_entry["orig_path"] = entry["stim_path"]  # keep original reference
        new_entry["stim_path"] = dst
        new_entry["filename"] = new_name
        updated.append(new_entry)

    return updated

meta_p1_updated = copy_and_update(meta_p1, dst_p1)
meta_p2_updated = copy_and_update(meta_p2, dst_p2)

# --- Save new metadata JSONs ---
with open(os.path.join(dst_p1, "filenames.json"), "w") as f:
    json.dump(meta_p1_updated, f, indent=2)

with open(os.path.join(dst_p2, "filenames.json"), "w") as f:
    json.dump(meta_p2_updated, f, indent=2)

# --- Summary ---
print(f"✅ Done. Saved:")
print(f"  {len(meta_p1_updated)} files to {dst_p1}")
print(f"  {len(meta_p2_updated)} files to {dst_p2}")
print(f"  Each half has {len(industrial_p1)} industrial and {len(nature_p1)} nature sounds.")

In [ ]:
# ========== FIX BAD STIM_PATH PREFIXES ==========
import json, os, datetime

# --- Backup first ---
timestamp = datetime.datetime.now().strftime("%Y-%m-%dT%H-%M-%S")
backup_path = os.path.join(TO_SAVE_PATH, f"filenames_backup_stimpathfix_{timestamp}.json")
with open(backup_path, "w") as f:
    json.dump(processed, f, indent=2)
print(f"🧷 Backup saved to {backup_path}")

# --- Define old and new prefixes ---
BAD_PREFIX = "/Users/bjm/Documents/other/NaturalSounds2025/"
GOOD_PREFIX = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/NaturalSounds2025/"

# --- Iterate and fix ---
n_fixed = 0
for entry in processed:
    stim = entry.get("stim_path", "")
    if stim.startswith(BAD_PREFIX):
        # replace the prefix and update filename reference
        new_stim = stim.replace(BAD_PREFIX, GOOD_PREFIX, 1)
        entry["stim_path"] = new_stim
        entry["filename"] = os.path.basename(new_stim)
        n_fixed += 1

# --- Save updated JSON ---
with open(json_path, "w") as f:
    json.dump(processed, f, indent=2)

print(f"✅ Fixed {n_fixed} stim_path entries in {json_path}")
print("🧩 All paths now point to the Mindhive directory.")